# GRU: Hotel Review Sentiment Classification
**SE4050 Deep Learning 2026 · Individual contribution: GRU (Gated Recurrent Unit)**

Task: classify a hotel review as **positive (1)** or **negative (0)**.
This notebook trains and evaluates the GRU under exactly the same conditions as the Simple RNN, LSTM and 1D CNN.

**Fairness and leakage rules followed in this notebook**

| Rule | How it is enforced |
|---|---|
| Same dataset, split, preprocessing, vocabulary, padding | Defined once in the helper cells of Step 0: the data pipeline, and a fixed list of validation and test row ids |
| Same training settings | Every hyperparameter except the GRU layers comes from the *Settings* cell in Step 0 |
| Vocabulary and class weights use the **training set only** | `build_vocab()` and `compute_class_weights()` only ever receive training rows |
| Test set stays unseen | Steps 0 to 8 never use it to fit or choose anything. It is used once for the main model in Step 9, after the model is finalised (and once for the optional Step 13 extension, after its design was fixed on validation data) |
| Same metrics for every model | The evaluation helpers in Step 0 |

## Step 0: Setup and libraries

| Library | Used for |
|---|---|
| `pandas`, `numpy` | Tables and arrays |
| `scikit-learn` | Evaluation metrics: accuracy, precision, recall, F1, ROC-AUC, confusion matrix |
| `tensorflow` / `keras` | The 1D CNN model (`Embedding`, `GRU`, `Dropout`, `Dense`), the optimiser and EarlyStopping |
| `matplotlib` | Figures |
| `re`, `collections` (standard library) | Text cleaning and word counts |

**Running in Google Colab?** Just run the cells from the top. When asked, choose `deceptive-opinion.csv` from your computer. No other file is needed. Colab's disk is temporary, so download the `results/` folder when you finish.

### Settings and helper code

The next four cells contain everything this notebook needs besides the dataset: imports, the group settings, and helper functions for the data pipeline, evaluation and plots.
**To keep the four models identical, copy these same cells unchanged into the Simple RNN, LSTM and 1D CNN notebooks.** Only the model layers should differ.

In [ ]:
# ---- Imports and dataset location ----------------------------------------------------------------
import sys, re, json, time, pathlib, textwrap
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tensorflow import keras
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

layers = keras.layers
IN_COLAB = "google.colab" in sys.modules

# The dataset is the only file this notebook needs.
DATA_NAME = "deceptive-opinion.csv"
here = pathlib.Path.cwd().resolve()
candidates = [here / DATA_NAME, here / "dataset" / DATA_NAME, here.parent / "dataset" / DATA_NAME,
              pathlib.Path("/content") / DATA_NAME]
DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None and IN_COLAB:
    from google.colab import files
    print(f"Please choose {DATA_NAME} from your computer...")
    uploaded = files.upload()                       # saved in the current folder
    DATA_PATH = here / next(iter(uploaded))
if DATA_PATH is None:
    raise FileNotFoundError(f"Cannot find {DATA_NAME}. Put it next to this notebook "
                            "(or inside a 'dataset' folder) and run this cell again.")

RESULTS = here / "results"                          # figures, metrics and the trained model are saved here
RESULTS.mkdir(exist_ok=True)
print("Dataset:", DATA_PATH)
print("Results folder:", RESULTS)

In [ ]:
# ---- Group settings: identical for Simple RNN, LSTM, GRU and 1D CNN ------------------------------
SEED = 42

# Labels: this dataset has no star rating, so the `polarity` column is the label.
LABEL_COLUMN = "polarity"
LABEL_MAP = {"negative": 0, "positive": 1}

# Text -> integer sequences
VOCAB_SIZE = 5000          # total ids, including <PAD>=0 and <OOV>=1
MAX_LEN = 300              # tokens per review after cleaning
PAD_TOKEN, OOV_TOKEN = "<PAD>", "<OOV>"
PADDING = "pre"            # zeros in front, so a recurrent layer ends on real words
TRUNCATING = "post"        # long reviews keep their first MAX_LEN tokens

# Training
EMBED_DIM = 64
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_EPOCHS = 30
PATIENCE = 5               # EarlyStopping patience, monitoring validation loss
MONITOR = "val_loss"
THRESHOLD = 0.5            # probability cut-off used for every model's test metrics

# Fixed 80/10/10 split (stratified on the label, seed 42, made after removing duplicates).
# Numbers are 0-based row positions in deceptive-opinion.csv. Every row not listed here is a TRAINING row.
# All four models must use exactly these two lists.
VAL_IDS = {
    3, 4, 6, 7, 14, 24, 25, 27, 33, 38, 46, 50, 60, 66, 72, 74, 77, 85, 89, 92, 106, 110, 112, 136,
    144, 172, 211, 215, 224, 230, 238, 242, 248, 252, 288, 302, 311, 319, 336, 345, 368, 376, 385,
    393, 402, 404, 434, 440, 442, 465, 473, 510, 512, 515, 517, 524, 533, 536, 542, 549, 553, 558,
    579, 581, 594, 603, 624, 630, 638, 644, 645, 661, 663, 665, 699, 713, 729, 754, 760, 788, 801,
    813, 820, 821, 827, 832, 840, 864, 873, 893, 897, 900, 904, 907, 908, 923, 930, 936, 962, 963,
    989, 991, 1002, 1008, 1017, 1043, 1045, 1054, 1073, 1098, 1112, 1119, 1134, 1143, 1170, 1189,
    1191, 1201, 1239, 1258, 1262, 1263, 1264, 1270, 1280, 1295, 1308, 1309, 1314, 1328, 1344, 1350,
    1359, 1360, 1364, 1367, 1368, 1369, 1370, 1377, 1378, 1381, 1418, 1423, 1450, 1467, 1479, 1480,
    1490, 1505, 1506, 1522, 1547, 1551, 1552, 1573, 1578, 1588, 1593, 1594
}
TEST_IDS = {
    0, 2, 5, 9, 39, 45, 47, 55, 108, 117, 120, 125, 138, 139, 145, 171, 191, 193, 213, 222, 227,
    228, 232, 256, 258, 260, 272, 273, 276, 281, 299, 314, 321, 325, 349, 362, 364, 391, 409, 424,
    436, 445, 467, 469, 484, 502, 503, 505, 507, 523, 525, 531, 545, 547, 572, 573, 605, 606, 612,
    622, 623, 633, 669, 671, 681, 687, 691, 698, 718, 725, 731, 741, 752, 753, 765, 770, 771, 779,
    784, 785, 804, 814, 834, 835, 847, 852, 859, 866, 882, 887, 889, 901, 932, 940, 958, 961, 968,
    972, 993, 1003, 1007, 1019, 1020, 1033, 1044, 1046, 1055, 1072, 1076, 1079, 1091, 1117, 1141,
    1147, 1149, 1151, 1176, 1182, 1183, 1193, 1195, 1196, 1205, 1210, 1217, 1222, 1231, 1259, 1265,
    1275, 1278, 1279, 1288, 1293, 1296, 1302, 1306, 1312, 1324, 1357, 1365, 1366, 1396, 1404, 1416,
    1446, 1451, 1465, 1478, 1485, 1487, 1492, 1503, 1504, 1516, 1524, 1540, 1562, 1582, 1597
}
keras.utils.set_random_seed(SEED)          # seeds Python, NumPy and TensorFlow
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| seed", SEED)
print("Devices:", [d.device_type for d in tf.config.list_physical_devices()])

In [ ]:
# ---- Data pipeline: load -> label -> de-duplicate -> split -> clean -> vocabulary -> pad ----------
# Leakage rule: the vocabulary and the class weights are built from TRAINING rows only.
# Validation and test rows are only ever encoded with the training vocabulary.

def load_labelled_data() -> pd.DataFrame:
    """Read the CSV, map polarity -> 0/1 and drop exact duplicate reviews.

    Duplicates must go before splitting, otherwise a copy of a training review could land in the
    test set and the test set would no longer be unseen.
    """
    df = pd.read_csv(DATA_PATH)
    df["row_id"] = np.arange(len(df))                   # 0-based row number in the CSV
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df[LABEL_COLUMN].map(LABEL_MAP)
    assert df["label"].notna().all(), "unexpected label value"
    df["label"] = df["label"].astype(int)
    df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
    return df[["row_id", "text", "label", "hotel", "deceptive", "source"]]


def assign_split(df: pd.DataFrame) -> np.ndarray:
    """'train' / 'val' / 'test' for every row, from the fixed id lists above."""
    if not (VAL_IDS | TEST_IDS) <= set(df["row_id"]):
        raise ValueError("The fixed validation/test ids do not match this dataset. Was the CSV changed?")
    return np.where(df["row_id"].isin(TEST_IDS), "test",
                    np.where(df["row_id"].isin(VAL_IDS), "val", "train"))


# Text cleaning: identical for all models; a pure per-review function, nothing is fitted.
_HTML = re.compile(r"<[^>]+>")
_URL = re.compile(r"(https?://\S+|www\.\S+)")
_CONTRACTIONS = [                       # keep negation: "didn't" -> "did not", not "didn t"
    (r"\bwon't\b", "will not"), (r"\bcan't\b", "can not"), (r"\bcannot\b", "can not"),
    (r"n't\b", " not"), (r"'re\b", " are"), (r"'ve\b", " have"), (r"'ll\b", " will"),
    (r"'d\b", " would"), (r"'m\b", " am"), (r"'s\b", ""),
]
_NON_LETTER = re.compile(r"[^a-z\s]")
_SPACES = re.compile(r"\s+")


def clean_text(text: str) -> str:
    """lowercase -> strip HTML -> strip URLs -> expand contractions -> keep letters only.

    Stop-words are deliberately NOT removed: words such as "not", "no" and "never" flip the sentiment.
    """
    text = text.lower().replace("’", "'")
    text = _HTML.sub(" ", text)
    text = _URL.sub(" ", text)
    for pattern, replacement in _CONTRACTIONS:
        text = re.sub(pattern, replacement, text)
    text = _NON_LETTER.sub(" ", text)           # digits, punctuation, symbols
    return _SPACES.sub(" ", text).strip()


def build_vocab(train_token_lists, vocab_size: int = VOCAB_SIZE) -> dict:
    """Most frequent TRAINING words -> ids. 0 = <PAD>, 1 = <OOV>, real words start at 2."""
    counts = Counter(tok for tokens in train_token_lists for tok in tokens)
    words = [w for w, _ in counts.most_common(vocab_size - 2)]
    vocab = {PAD_TOKEN: 0, OOV_TOKEN: 1}
    vocab.update({w: i + 2 for i, w in enumerate(words)})
    return vocab


def encode(tokens, vocab: dict) -> list:
    oov = vocab[OOV_TOKEN]
    return [vocab.get(t, oov) for t in tokens]


def decode(ids, vocab: dict, skip_pad: bool = True) -> list:
    """Inverse of `encode`: ids -> words. Unknown words come back as <OOV> (the original is lost)."""
    inverse = {i: w for w, i in vocab.items()}
    return [inverse[int(i)] for i in ids if not (skip_pad and int(i) == 0)]


def pad(sequences, max_len: int = MAX_LEN) -> np.ndarray:
    out = np.zeros((len(sequences), max_len), dtype="int32")
    for i, seq in enumerate(sequences):
        seq = seq[:max_len] if TRUNCATING == "post" else seq[-max_len:]
        if PADDING == "pre":
            out[i, max_len - len(seq):] = seq
        else:
            out[i, :len(seq)] = seq
    return out


def compute_class_weights(y_train) -> dict:
    """weight_c = n_samples / (n_classes * n_samples_in_class_c)   (same as sklearn 'balanced')."""
    classes, counts = np.unique(y_train, return_counts=True)
    return {int(c): float(len(y_train) / (len(classes) * n)) for c, n in zip(classes, counts)}


@dataclass
class Data:
    X_train: np.ndarray
    y_train: np.ndarray
    X_val: np.ndarray
    y_val: np.ndarray
    X_test: np.ndarray
    y_test: np.ndarray
    vocab: dict
    class_weight: dict
    train_df: pd.DataFrame     # raw + cleaned text, kept for EDA / error analysis
    val_df: pd.DataFrame
    test_df: pd.DataFrame


def prepare_data() -> Data:
    """Everything a model needs, in one call."""
    df = load_labelled_data()
    df["split"] = assign_split(df)
    df["clean"] = df["text"].map(clean_text)
    df["tokens"] = df["clean"].str.split()

    parts = {name: g.reset_index(drop=True) for name, g in df.groupby("split")}
    vocab = build_vocab(parts["train"]["tokens"].tolist())          # <- TRAIN ONLY

    def to_xy(part: pd.DataFrame):
        seqs = [encode(tokens, vocab) for tokens in part["tokens"]]
        return pad(seqs), part["label"].to_numpy(dtype="int32")

    X_train, y_train = to_xy(parts["train"])
    X_val, y_val = to_xy(parts["val"])
    X_test, y_test = to_xy(parts["test"])
    return Data(X_train, y_train, X_val, y_val, X_test, y_test, vocab,
                compute_class_weights(y_train), parts["train"], parts["val"], parts["test"])

In [ ]:
# ---- Evaluation helpers: the same metrics and result format for all four models ------------------
# Positive class (label 1) is the class used for precision / recall / F1.

def compute_metrics(y_true, y_prob, threshold: float = THRESHOLD) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        # per-class view, needed for the error analysis
        "negative_precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "negative_recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "negative_f1": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def bootstrap_ci(y_true, y_prob, n_boot: int = 2000, seed: int = SEED) -> dict:
    """95% bootstrap interval for accuracy / F1 / ROC-AUC on the test set.
    It only describes how uncertain the (small) test set makes the numbers. It is NOT used to tune anything."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).ravel()
    rng = np.random.default_rng(seed)
    out = {"accuracy": [], "f1": [], "roc_auc": []}
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        m = compute_metrics(y_true[idx], y_prob[idx])
        for k in out:
            out[k].append(m[k])
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in out.items()}


def save_result(path, model_name: str, test_metrics: dict, *, training_time_sec: float,
                trainable_params: int, total_params: int, epochs_run: int, best_epoch: int,
                extra: dict = None) -> dict:
    """One JSON file per model with the same keys for every model, for the group's comparison table."""
    row = {
        "model": model_name,
        "accuracy": test_metrics["accuracy"], "precision": test_metrics["precision"],
        "recall": test_metrics["recall"], "f1": test_metrics["f1"], "roc_auc": test_metrics["roc_auc"],
        "training_time_sec": training_time_sec,
        "trainable_params": trainable_params, "total_params": total_params,
        "epochs_run": epochs_run, "best_epoch": best_epoch,
        "seed": SEED, "max_len": MAX_LEN, "vocab_size": VOCAB_SIZE,
        "embed_dim": EMBED_DIM, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "threshold": THRESHOLD,
        "confusion_matrix": {k: test_metrics[k] for k in ("tn", "fp", "fn", "tp")},
    }
    if extra:
        row.update(extra)
    with open(path, "w") as f:
        json.dump(row, f, indent=2)
    return row


# ---- Plot helpers: same figure style for all models -----------------------------------------------
# White background, thin lines, light grid, no top/right frame, text in neutral ink, legend for 2+ series.
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e5e4e0"
BLUE_RAMP = LinearSegmentedColormap.from_list("blue_seq", ["#cde2fb", "#3987e5", "#0d366b"])

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "legend.frameon": False, "lines.linewidth": 2,
})


def save_and_show(fig, path=None, show=True):
    fig.tight_layout()
    if path:
        fig.savefig(path)
    if show:
        plt.show()
    plt.close(fig)


def plot_curve(history: dict, metric: str, model_name: str, best_epoch=None, path=None, show=True):
    """Training vs validation curve for 'accuracy' or 'loss'."""
    train, val = history[metric], history["val_" + metric]
    epochs = np.arange(1, len(train) + 1)
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(epochs, train, color=BLUE, label="Training")
    ax.plot(epochs, val, color=ORANGE, label="Validation")
    for series, color, name in ((train, BLUE, "Training"), (val, ORANGE, "Validation")):
        ax.plot(epochs[-1], series[-1], "o", color=color, markersize=6,
                markeredgecolor="white", markeredgewidth=1.5)
        ax.annotate(name, (epochs[-1], series[-1]), xytext=(8, 0), textcoords="offset points",
                    va="center", color=INK2, fontsize=10)
    if best_epoch:
        ax.axvline(best_epoch, color=INK2, linestyle=":", linewidth=1.2)
        ax.text(best_epoch, ax.get_ylim()[1], f" best epoch {best_epoch}", color=INK2,
                fontsize=9, va="top", ha="left")
    ax.set_xlim(0.7, epochs[-1] + 0.15 * max(epochs[-1], 6))
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric.capitalize())
    ax.set_title(f"{model_name}: training vs validation {metric}", loc="left")
    ax.legend(loc="best")
    save_and_show(fig, path, show)


def plot_confusion_matrix(cm: np.ndarray, model_name: str, path=None, show=True):
    """2x2 confusion matrix (rows = true class, columns = predicted class)."""
    labels = ["Negative", "Positive"]
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    ax.imshow(cm, cmap=BLUE_RAMP, vmin=0, vmax=cm.max())
    ax.grid(False)
    for i in range(2):
        for j in range(2):
            dark = cm[i, j] > cm.max() * 0.55
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=18,
                    fontweight="bold", color="white" if dark else INK)
    ax.set_xticks([0, 1], labels)
    ax.set_yticks([0, 1], labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{model_name}: confusion matrix (test)", loc="left")
    for s in ax.spines.values():
        s.set_visible(False)
    save_and_show(fig, path, show)


def plot_roc(y_true, y_prob, auc: float, model_name: str, threshold: float = 0.5, path=None, show=True):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5.2, 4.8))
    ax.plot([0, 1], [0, 1], color=INK2, linestyle="--", linewidth=1.2)
    ax.text(0.55, 0.47, "chance (AUC = 0.50)", color=INK2, fontsize=9, rotation=33)
    ax.plot(fpr, tpr, color=BLUE)
    k = np.argmin(np.abs(thr - threshold))          # operating point at the 0.5 cut-off
    ax.plot(fpr[k], tpr[k], "o", color=ORANGE, markersize=8, markeredgecolor="white", markeredgewidth=1.5)
    ax.annotate(f"threshold {threshold}", (fpr[k], tpr[k]), xytext=(10, -14),
                textcoords="offset points", color=INK2, fontsize=9)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"{model_name}: ROC curve (test), AUC = {auc:.3f}", loc="left")
    ax.set_xlim(-0.01, 1.01)
    ax.set_ylim(-0.01, 1.01)
    ax.set_aspect("equal")
    save_and_show(fig, path, show)


def plot_probability_hist(y_true, y_prob, model_name: str, threshold: float = 0.5, path=None, show=True):
    """How confident is the model? Predicted P(positive), split by the true class (stacked bars)."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob).ravel()
    bins = np.linspace(0, 1, 21)
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.hist([y_prob[y_true == 1], y_prob[y_true == 0]], bins=bins, stacked=True,
            color=[BLUE, ORANGE], label=["True positive reviews", "True negative reviews"],
            edgecolor="white", linewidth=1.5)
    ax.axvline(threshold, color=INK2, linestyle=":", linewidth=1.2)
    ax.set_xlabel("Predicted probability of positive")
    ax.set_ylabel("Number of test reviews")
    ax.set_title(f"{model_name}: confidence on the test set", loc="left")
    ax.legend(loc="upper center")
    save_and_show(fig, path, show)

## Step 1: Data audit (cleaning checks on the raw file)

Before modelling we check the raw CSV for **missing values, duplicates, encoding problems and outliers**.
These are hygiene checks. They learn nothing from the data, and the only rows removed are exact duplicate reviews, so they are run on the whole file before the split and every model gets the same result.

In [ ]:
raw = pd.read_csv(DATA_PATH)                       # the file exactly as given
text = raw["text"].astype(str).str.strip()
cleaned = text.map(clean_text)

print(f"Rows: {len(raw):,} | columns: {list(raw.columns)}\n")
print("MISSING VALUES / BLANK TEXT")
print(pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing": raw.isna().sum(),
                    "blank": [(raw[c].astype(str).str.strip() == "").sum() for c in raw.columns]}).to_string())
print("\nLabel values found:", raw["polarity"].value_counts().to_dict(), "-> encoded with", LABEL_MAP)

print("\nDUPLICATES")
print("  exact duplicate rows (all columns):", raw.duplicated().sum())
print("  duplicate review texts:", text.duplicated().sum())
print("  duplicates after cleaning (ignoring case / punctuation / digits):", cleaned.duplicated().sum())
print("  duplicate texts with conflicting labels:", raw.assign(t=text).groupby("t")["polarity"].nunique().gt(1).sum())
print("  extra reviews that only share their first 200 characters with another review:",
      cleaned.str[:200].duplicated().sum() - cleaned.duplicated().sum())

print("\nENCODING")
print("  reviews with non-ASCII characters:", sum(any(ord(ch) > 127 for ch in s) for s in text))
print("  replacement characters (U+FFFD):", text.str.contains("\ufffd").sum(),
      "| mojibake patterns:", text.str.contains("Ã|â€|Â").sum())
print("  reviews containing HTML tags:", text.str.contains(r"<[^>]+>").sum(),
      "| URLs:", text.str.contains(r"https?://|www\.").sum(), "| digits:", text.str.contains(r"\d").sum())

In [ ]:
tokens = cleaned.str.split().map(len)
q1, q3 = tokens.quantile([0.25, 0.75])
lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
print("OUTLIERS: review length in tokens (IQR rule)")
print(f"  min {tokens.min()} | Q1 {q1:.0f} | median {tokens.median():.0f} | Q3 {q3:.0f} | max {tokens.max()}")
print(f"  fences: lower {lower:.0f}, upper {upper:.0f} -> {int((tokens < lower).sum())} reviews below, "
      f"{int((tokens > upper).sum())} above ({100 * (tokens > upper).mean():.1f}%)")
print(f"  reviews that are empty after cleaning: {int((tokens == 0).sum())}")

fig, ax = plt.subplots(figsize=(7, 2.9))
groups = [tokens[raw["polarity"] == k] for k in ("positive", "negative")]
box_style = dict(widths=0.55, patch_artist=True,
                 flierprops=dict(marker="o", markersize=3, markerfacecolor=INK2, markeredgecolor="none", alpha=0.5),
                 medianprops=dict(color="white", linewidth=2), whiskerprops=dict(color=INK2), capprops=dict(color=INK2))
try:
    bp = ax.boxplot(groups, orientation="horizontal", **box_style)
except TypeError:                                   # older matplotlib (before 3.10)
    bp = ax.boxplot(groups, vert=False, **box_style)
for box, colour in zip(bp["boxes"], (BLUE, ORANGE)):
    box.set(facecolor=colour, edgecolor="none")
ax.set_yticks([1, 2], ["Positive", "Negative"])
ax.set_xlabel("Tokens per review (dots = outliers by the IQR rule)")
ax.set_title("Review length outliers", loc="left")
save_and_show(fig, RESULTS / "cnn1d_audit_length_outliers.png")

**What we do about each issue**

| Check | Action |
|---|---|
| Missing values / blank text | None needed if the counts above are zero. Otherwise such rows would be dropped for all models |
| Duplicates | Exact duplicate reviews are dropped (first copy kept) in `load_labelled_data()`, **before** the split, so a copy of a training review cannot appear in the test set |
| Encoding | The file is read as UTF-8. Curly apostrophes are normalised and all remaining non-letters are removed by the cleaning step |
| Outliers | **Kept.** Long reviews are real opinions and are cut to `MAX_LEN` in Step 4. Short reviews are still valid. Removing outliers would change what the test set contains and make the four models incomparable |

## Step 2: Data, labels and the 80/10/10 split

* Dataset: `dataset/deceptive-opinion.csv`, 1,600 hotel reviews of 20 Chicago hotels.
* This file has **no star rating**, so the rating rule (≥4 positive, <3 negative, 3 removed) cannot be applied. The label is the file's own `polarity` column: **positive = 1, negative = 0**. There are no neutral reviews to remove. *The group must confirm that all four models use this same label.*
* Exact duplicate reviews (found in Step 1) are removed **before** splitting. Otherwise a copy of a training review could end up in the test set.
* The split is stratified 80% / 10% / 10% and fixed by the row-id lists in the *Settings* cell, so all four models see identical rows.

In [ ]:
data = prepare_data()

def split_table(name, y):
    return {"split": name, "reviews": len(y), "positive": int(y.sum()),
            "negative": int((1 - y).sum()), "% positive": round(100 * y.mean(), 1)}

pd.DataFrame([split_table("train (80%)", data.y_train),
              split_table("validation (10%)", data.y_val),
              split_table("test (10%)", data.y_test)])

## Step 3: Exploratory data analysis (training set)

Text analysis uses the **training** set only, so nothing in the validation or test text can influence our choices.

In [ ]:
splits = {"train": data.y_train, "validation": data.y_val, "test": data.y_test}
x, w = np.arange(len(splits)), 0.38
pos = [int(y.sum()) for y in splits.values()]
neg = [int((1 - y).sum()) for y in splits.values()]
fig, ax = plt.subplots(figsize=(6.5, 3.8))
b1 = ax.bar(x - w / 2, pos, w - 0.04, color=BLUE, label="Positive (1)")
b2 = ax.bar(x + w / 2, neg, w - 0.04, color=ORANGE, label="Negative (0)")
for bars in (b1, b2):
    ax.bar_label(bars, padding=3, color=INK2, fontsize=10)
ax.set_xticks(x, list(splits))
ax.set_ylabel("Reviews")
ax.set_ylim(0, max(pos + neg) * 1.12)
ax.set_title("Class balance in each split", loc="left")
ax.legend(loc="upper right")
save_and_show(fig, RESULTS / "cnn1d_eda_class_balance.png")

In [ ]:
from collections import Counter

tr = data.train_df.assign(n_tokens=lambda d: d["tokens"].map(len),
                          label_name=lambda d: d["label"].map({1: "positive", 0: "negative"}))
print("Tokens per review by class (training set):")
print(tr.groupby("label_name")["n_tokens"].describe()[["count", "mean", "50%", "max"]].round(1).to_string())

NEGATION = {"not", "no", "never", "nothing", "nobody", "none", "without"}
has_negation = tr["tokens"].map(lambda t: bool(NEGATION & set(t)))
share = tr.assign(neg=has_negation).groupby("label_name")["neg"].mean().mul(100).round(1)
print("\n% of reviews containing a negation word:", share.to_dict(),
      "\n(negations are kept, because they change the meaning of a sentence)")

print("\nReview origin x label (training set). 'deceptive' is not a model input, it is only used for error analysis:")
print(pd.crosstab(tr["label_name"], tr["deceptive"]).to_string())
print("\nMost frequent tokens (stop-words included):",
      [w for w, _ in Counter(t for toks in tr["tokens"] for t in toks).most_common(12)])

In [ ]:
# Could the model "cheat" using review length alone? Negative reviews are longer, so check how far a
# one-number length rule gets. Threshold chosen on TRAIN, scored on VALIDATION (test is not touched).
from sklearn.metrics import roc_auc_score

len_train = data.train_df["tokens"].map(len).to_numpy()
len_val = data.val_df["tokens"].map(len).to_numpy()
candidates = np.unique(len_train)
train_acc = [((len_train <= t).astype(int) == data.y_train).mean() for t in candidates]
T = candidates[int(np.argmax(train_acc))]
print(f"Length-only rule 'longer than {T} tokens -> negative': train accuracy {max(train_acc):.3f}, "
      f"validation accuracy {((len_val <= T).astype(int) == data.y_val).mean():.3f}, "
      f"validation ROC-AUC {roc_auc_score(data.y_val, -len_val):.3f}  (chance = 0.500)")

In [ ]:
# Which words separate positive from negative reviews? (display only: common stop-words hidden here,
# but they are NOT removed from the model input)
STOP = set("a an the and or but of to in on at for with from by as is was were are be been am it its this that "
           "these those i me my we our you your they their he she him her them us so very just also than then "
           "there here had have has do did does would could will".split())
docs = {k: [set(t) - STOP for t in tr.loc[tr["label"] == k, "tokens"]] for k in (1, 0)}
n_docs = {k: len(v) for k, v in docs.items()}
doc_freq = {k: Counter(word for d in docs[k] for word in d) for k in (1, 0)}
score = {word: np.log((doc_freq[1][word] + 1) / (n_docs[1] + 2)) - np.log((doc_freq[0][word] + 1) / (n_docs[0] + 2))
         for word in set(doc_freq[1]) | set(doc_freq[0]) if doc_freq[1][word] + doc_freq[0][word] >= 20}
ranked = sorted(score.items(), key=lambda kv: kv[1])
panels = ((ranked[:12], ORANGE, "Words that lean negative"), (ranked[::-1][:12], BLUE, "Words that lean positive"))

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.3))
for ax, (items, colour, title) in zip(axes, panels):
    words, values = zip(*items)
    ax.barh(np.arange(len(words)), np.abs(values), color=colour, height=0.65)
    ax.set_yticks(np.arange(len(words)), words)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("log-odds (strength of association)")
    ax.set_title(title, loc="left", fontsize=11)
save_and_show(fig, RESULTS / "cnn1d_eda_distinctive_words.png")

## Step 4: Feature engineering (turning text into numbers)

A neural network needs numbers, not words. For a deep-learning text model, feature engineering means **building the numeric input**.
We do **not** hand-craft extra features (word counts, sentiment scores, and so on): the embedding and the GRU learn their own features from the words, and all four models must receive exactly the same input.

| Stage | What happens | Fitted on |
|---|---|---|
| 1. Cleaning | lowercase, remove HTML / URLs / digits / punctuation, keep negations | nothing (fixed rules) |
| 2. Tokenisation | split the cleaned text on spaces | nothing |
| 3. Vocabulary | the `VOCAB_SIZE - 2` most frequent words, plus `<PAD>` = 0 and `<OOV>` = 1 | **training set only** |
| 4. Integer encoding (word to id) | unknown words become `<OOV>` | uses the training vocabulary |
| 5. Padding / truncation | every review becomes exactly `MAX_LEN` ids: zeros in front, long reviews cut at the end | uses `MAX_LEN` |
| 6. Embedding (inside the model) | id to a learned 64-number vector | learned during training |

In [ ]:
sample = data.train_df.iloc[3]
ids = encode(sample["tokens"], data.vocab)
padded = data.X_train[3]                              # the same review as it enters the model

print("RAW     :", sample["text"][:160].replace("\n", " "))
print("CLEAN   :", sample["clean"][:160])
print("TOKENS  :", sample["tokens"][:10])
print("IDS     :", ids[:10])
print("DECODED :", decode(ids[:10], data.vocab), "  <- ids back to words (decoding)")
print(f"PADDED  : {len(padded)} ids, {int((padded == 0).sum())} leading zeros, last 8 ids = {padded[-8:].tolist()}")
assert decode(padded, data.vocab) == decode(ids[:MAX_LEN], data.vocab)   # padding lost nothing
print("Round trip OK: decode(padded input) equals the encoded review (first MAX_LEN tokens)")
print("\nNegation is kept:", clean_text("The room wasn't clean and we can't recommend it!"))

In [ ]:
train_len = data.train_df["tokens"].map(len)
print(f"Tokens per training review: median {int(train_len.median())}, "
      f"95th percentile {int(np.percentile(train_len, 95))}, max {train_len.max()}")
print(f"MAX_LEN = {MAX_LEN}: {100*(train_len > MAX_LEN).mean():.1f}% of training reviews are truncated")

counts = Counter(t for toks in data.train_df["tokens"] for t in toks)
covered = sum(c for _, c in counts.most_common(VOCAB_SIZE - 2)) / sum(counts.values())
val_tokens = [t for toks in data.val_df["tokens"] for t in toks]     # validation only, for the OOV rate
val_oov = np.mean([t not in data.vocab for t in val_tokens])
print(f"Unique training words: {len(counts):,}. VOCAB_SIZE = {VOCAB_SIZE:,} covers "
      f"{100*covered:.1f}% of training tokens; {100*val_oov:.1f}% of validation tokens map to <OOV>")

for name, X, y in [("train", data.X_train, data.y_train), ("validation", data.X_val, data.y_val), ("test", data.X_test, data.y_test)]:
    print(f"  {name:10s} X {X.shape} {X.dtype} | y {y.shape}")
print(f"Share of padding zeros in the training matrix: {100 * (data.X_train == 0).mean():.1f}%")

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(train_len, bins=40, color=BLUE, edgecolor="white", linewidth=1)
ax.axvline(MAX_LEN, color=ORANGE, linewidth=2)
ax.text(MAX_LEN + 8, ax.get_ylim()[1] * 0.9, f"MAX_LEN = {MAX_LEN}", color=INK2)
ax.set_xlabel("Tokens per review (training set)"); ax.set_ylabel("Number of reviews")
ax.set_title("Review length distribution", loc="left")
save_and_show(fig, RESULTS / "cnn1d_eda_review_length.png")

## Step 5: Class weights

Class weights make mistakes on a rare class cost more during training, so the model does not simply learn to favour the majority class. We use the group's formula (the same as scikit-learn's `class_weight="balanced"`):

`weight_c = n_samples / (n_classes × n_samples_in_class_c)`

**In this dataset the classes are (almost) perfectly balanced**, so the weights come out at about 1.0 and have essentially no effect. The step is kept anyway so all four models follow the same, imbalance-safe procedure. It would matter on the ~20k-review rating-based hotel dataset, which has many more positive than negative reviews.

In [ ]:
print("Class weights (from the training labels only):", {k: round(v, 4) for k, v in data.class_weight.items()})

## Step 6: The 1D CNN model

```
Input (300 token ids)
  → Embedding(5000, 64)        each word id → a learned 64-number vector
  → GRU(64)                    reads the sequence word by word, returns its final memory
  → Dropout(0.5)
  → Dense(32, ReLU)
  → Dropout(0.3)
  → Dense(1, Sigmoid)          probability that the review is positive
```

| Component | Choice | Why |
|---|---|---|
| **Embedding** | `Embedding(VOCAB_SIZE, 64)`, trained from scratch | A word id is just an arbitrary number. The embedding turns it into a dense vector where words used in similar ways (e.g. *dirty*, *filthy*) end up close together. No pretrained vectors are used, which keeps the project simple and the comparison fair. |
| **Embedding dim 64** | Small | Only ~1,276 training reviews. A larger dimension means more parameters to learn from little data and more overfitting. |
| **GRU(64)** | 64 hidden units | Enough memory to track sentiment cues over a ~150-word review, but small enough not to memorise 1,276 reviews. Default `tanh` activation and `sigmoid` gates. |
| **Dropout 0.5 after the GRU** | Randomly zeroes 50% of the GRU outputs while training | The strongest regulariser, placed where the model is most likely to memorise. |
| **Dense(32, ReLU)** | One small hidden layer | Lets the network combine the 64 GRU features non-linearly before the decision. ReLU is simple, fast and avoids vanishing gradients. |
| **Dropout 0.3 after the Dense layer** | Lighter dropout | The layer is small, so we drop less. |
| **Dense(1, Sigmoid)** | One output in (0, 1) | Binary problem. The sigmoid output is read as P(positive), and pairs with binary cross-entropy. |

**Why a GRU for reviews?** A review is an ordered sequence, and meaning depends on order ("*not* clean" ≠ "clean"). A GRU reads word by word and keeps a memory that it can update or partly forget through its gates, so it can carry sentiment cues across a long review while training more easily than a Simple RNN.

In [ ]:
def build_cnn1d():
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(VOCAB_SIZE, EMBED_DIM),
        layers.Conv1D(128, 5, activation="relu"),
        layers.GlobalMaxPooling1D(),
        layers.Dropout(0.5),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ], name="cnn1d_sentiment")

model = build_cnn1d()
model.summary()

def n_params(weights):
    return int(sum(np.prod(w.shape) for w in weights))

breakdown = {layer.name: n_params(layer.trainable_weights) for layer in model.layers if layer.trainable_weights}
print("\nTrainable parameters per layer:", breakdown)
print(f"Embedding share of all parameters: {100 * breakdown['embedding'] / model.count_params():.1f}%")

## Step 7: Training configuration

| Setting | Value | Reason |
|---|---|---|
| Optimizer | Adam, learning rate 1e-3 | Adaptive optimiser, works well with default settings. Same for all four models |
| Loss | Binary cross-entropy | Standard loss for a two-class problem with a sigmoid output |
| Batch size | 32 | ~40 weight updates per epoch on 1,276 reviews |
| Max epochs | 30 | An upper limit. EarlyStopping normally stops much sooner |
| EarlyStopping | monitor **validation loss**, patience 5, `restore_best_weights=True` | Stops when validation loss has not improved for 5 epochs and rolls back to the best epoch, which limits overfitting |
| Class weights | from Step 5 | Same strategy as the other models |

Training uses the **training set** for weight updates and the **validation set** for monitoring. **The test set is not used here.**

In [ ]:
def train_cnn1d(seed, verbose=2):
    keras.utils.set_random_seed(seed)
    m = build_cnn1d()
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
              loss="binary_crossentropy", metrics=["accuracy"])
    early_stop = keras.callbacks.EarlyStopping(monitor=MONITOR, patience=PATIENCE,
                                               restore_best_weights=True)
    start = time.perf_counter()
    hist = m.fit(data.X_train, data.y_train,
                 validation_data=(data.X_val, data.y_val),          # validation only, never test
                 epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                 class_weight=data.class_weight, callbacks=[early_stop], verbose=verbose)
    return m, hist.history, time.perf_counter() - start

model, history, train_time = train_cnn1d(SEED)

epochs_run = len(history["loss"])
best_epoch = int(np.argmin(history["val_loss"])) + 1
pd.DataFrame(history).rename_axis("epoch").reset_index().assign(epoch=lambda d: d.epoch + 1) \
  .to_csv(RESULTS / "cnn1d_history.csv", index=False)
print(f"\nEpochs completed: {epochs_run} (max {MAX_EPOCHS}) | best epoch (lowest val loss): {best_epoch}")
print(f"Training time: {train_time:.1f} s ({train_time / epochs_run:.1f} s/epoch)")

## Step 8: Learning curves (training vs validation)

How to read them:
* **Both curves improve and end close together** → the model is learning and generalising.
* **Training keeps improving but validation loss turns upward** → overfitting. EarlyStopping restores the epoch before that.
* **Both stay poor** → underfitting.

In [ ]:
plot_curve(history, "accuracy", "1D CNN", best_epoch, RESULTS / "cnn1d_accuracy_curve.png")
plot_curve(history, "loss", "1D CNN", best_epoch, RESULTS / "cnn1d_loss_curve.png")

b = best_epoch - 1
print(f"At the best epoch ({best_epoch}):  train acc {history['accuracy'][b]:.3f} | val acc {history['val_accuracy'][b]:.3f} "
      f"| train loss {history['loss'][b]:.3f} | val loss {history['val_loss'][b]:.3f}")
print(f"Last epoch run ({epochs_run}):     train acc {history['accuracy'][-1]:.3f} | val acc {history['val_accuracy'][-1]:.3f} "
      f"| train loss {history['loss'][-1]:.3f} | val loss {history['val_loss'][-1]:.3f}")
print(f"Train-validation accuracy gap at best epoch: {history['accuracy'][b] - history['val_accuracy'][b]:+.3f}")

## Step 9: Final evaluation on the TEST set

> **Run this step once, after the model is finalised.** Nothing above depended on the test set. From here on, no hyperparameter, architecture or threshold may be changed based on these numbers.

Metrics use the group's shared definitions: threshold 0.5, and precision / recall / F1 for the **positive** class. ROC-AUC uses the raw probabilities.

In [ ]:
y_prob = model.predict(data.X_test, verbose=0).ravel()
test = compute_metrics(data.y_test, y_prob)

trainable = n_params(model.trainable_weights)
total = model.count_params()

print(f"TEST  accuracy {test['accuracy']:.4f} | precision {test['precision']:.4f} | recall {test['recall']:.4f} "
      f"| F1 {test['f1']:.4f} | ROC-AUC {test['roc_auc']:.4f}")
print(f"Confusion matrix:  TN={test['tn']}  FP={test['fp']}  FN={test['fn']}  TP={test['tp']}")
print(f"Trainable parameters: {trainable:,} (total {total:,}) | training time {train_time:.1f} s | epochs {epochs_run}")

ci = bootstrap_ci(data.y_test, y_prob)
print("\n95% bootstrap interval on the 160-review test set: " +
      " | ".join(f"{k} {lo:.3f}-{hi:.3f}" for k, (lo, hi) in ci.items()))

In [ ]:
cm = np.array([[test["tn"], test["fp"]], [test["fn"], test["tp"]]])
plot_confusion_matrix(cm, "1D CNN", RESULTS / "cnn1d_confusion_matrix.png")
plot_roc(data.y_test, y_prob, test["roc_auc"], "1D CNN", THRESHOLD, RESULTS / "cnn1d_roc_curve.png")
plot_probability_hist(data.y_test, y_prob, "1D CNN", THRESHOLD, RESULTS / "cnn1d_probability_hist.png")

In [ ]:
# Save everything the group needs for the final comparison table
model.save(RESULTS / "cnn1d_model.keras")
row = save_result(RESULTS / "cnn1d_metrics.json", "1D CNN", test, training_time_sec=round(train_time, 2),
                    trainable_params=trainable, total_params=total, epochs_run=epochs_run,
                    best_epoch=best_epoch, extra={"bootstrap_95ci": ci,
                    "class_weight": data.class_weight})

pred_df = data.test_df[["row_id", "label"]].assign(prob_positive=y_prob, pred=(y_prob >= THRESHOLD).astype(int))
pred_df.to_csv(RESULTS / "cnn1d_test_predictions.csv", index=False)

print("Row for the group comparison table:\n")
print("| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC | Training Time | Parameters |")
print("|---|---|---|---|---|---|---|---|")
print(f"| GRU | {test['accuracy']:.4f} | {test['precision']:.4f} | {test['recall']:.4f} | {test['f1']:.4f} "
      f"| {test['roc_auc']:.4f} | {train_time:.1f} s | {trainable:,} |")

## Step 10: Error analysis

This step **only describes** the GRU's mistakes for the report. Nothing here is used to change the model.
With only 160 test reviews there are few errors, so patterns are suggestive, not proof. Counts are shown next to every rate.

In [ ]:
err = data.test_df.copy()
err["prob"] = y_prob
err["pred"] = (y_prob >= THRESHOLD).astype(int)
err["outcome"] = np.select([(err.label == 1) & (err.pred == 1), (err.label == 0) & (err.pred == 0),
                            (err.label == 0) & (err.pred == 1), (err.label == 1) & (err.pred == 0)],
                           ["TP", "TN", "FP", "FN"], default="")
err["wrong"] = err.label != err.pred
err["n_tokens"] = err["tokens"].map(len)
err["confidence"] = np.where(err.pred == 1, err.prob, 1 - err.prob)

print(err["outcome"].value_counts().reindex(["TP", "TN", "FP", "FN"]).to_string())
print()
per_class = pd.DataFrame({
    "class": ["Negative (0)", "Positive (1)"],
    "reviews": [(err.label == 0).sum(), (err.label == 1).sum()],
    "precision": [test["negative_precision"], test["precision"]],
    "recall": [test["negative_recall"], test["recall"]],
    "F1": [test["negative_f1"], test["f1"]]}).round(3)
print(per_class.to_string(index=False))

In [ ]:
def rate_table(by, label):
    g = err.groupby(by)["wrong"].agg(reviews="size", errors="sum")
    g["error rate"] = (g["errors"] / g["reviews"]).round(3)
    g.index.name = label
    return g

print(rate_table("deceptive", "review origin"), "\n")
print(rate_table(err["n_tokens"] > MAX_LEN, f"longer than MAX_LEN={MAX_LEN}"), "\n")
print(rate_table(pd.cut(err["n_tokens"], [0, 100, 200, 10_000], labels=["<=100", "101-200", ">200"]),
                 "tokens"), "\n")
print("Mean model confidence   correct:", round(err.loc[~err.wrong, "confidence"].mean(), 3),
      "| wrong:", round(err.loc[err.wrong, "confidence"].mean(), 3))
print("Mean review length      correct:", round(err.loc[~err.wrong, "n_tokens"].mean(), 1),
      "| wrong:", round(err.loc[err.wrong, "n_tokens"].mean(), 1), "tokens")

In [ ]:
def show(outcome, k=5):
    rows = err[err.outcome == outcome].sort_values("confidence", ascending=False).head(k)
    title = {"FP": "FALSE POSITIVES (truly negative, predicted positive)",
             "FN": "FALSE NEGATIVES (truly positive, predicted negative)"}[outcome]
    print("=" * 100, f"\n{title}: showing the {len(rows)} most confident mistakes\n" + "=" * 100)
    for _, r in rows.iterrows():
        print(f"[{r.hotel} | {r.deceptive} | P(positive)={r.prob:.2f} | {r.n_tokens} tokens]")
        print(textwrap.fill(r.text.replace("\n", " ")[:600], 100), "\n")

show("FP")
show("FN")

## Step 11: Robustness check across random seeds (optional)

The test set has only 160 reviews, so one training run's score depends noticeably on the random start. Here the **same split and settings** are trained again with 5 other seeds.
This does **not** choose a better model. The headline result stays the seed-42 run above, which was fixed before any test result was seen. It only shows how much the numbers move from run to run.

In [ ]:
runs = [{"seed": SEED, "epochs": epochs_run, "val_acc": history["val_accuracy"][best_epoch - 1],
         "test_acc": test["accuracy"], "test_f1": test["f1"], "test_auc": test["roc_auc"]}]
for s in [1, 2, 3, 4, 5]:
    m, h, _ = train_cnn1d(s, verbose=0)
    p = m.predict(data.X_test, verbose=0).ravel()
    t = compute_metrics(data.y_test, p)
    runs.append({"seed": s, "epochs": len(h["loss"]), "val_acc": h["val_accuracy"][int(np.argmin(h["val_loss"]))],
                 "test_acc": t["accuracy"], "test_f1": t["f1"], "test_auc": t["roc_auc"]})
    keras.backend.clear_session()

seed_df = pd.DataFrame(runs)
print(seed_df.round(4).to_string(index=False))
print("\nmean ± std over", len(seed_df), "runs:")
for c in ["val_acc", "test_acc", "test_f1", "test_auc"]:
    print(f"  {c:9s} {seed_df[c].mean():.4f} ± {seed_df[c].std():.4f}")
seed_df.to_csv(RESULTS / "cnn1d_seed_robustness.csv", index=False)

## Step 12: Hand-over to the group

Files written to `cnn1d/results/`:

| File | Use |
|---|---|
| `cnn1d_metrics.json` | The GRU row for the comparison table (same schema for all models) |
| `cnn1d_test_predictions.csv` | Per-review test predictions, so the group can compare models review by review |
| `cnn1d_history.csv` | Epoch-by-epoch training and validation loss and accuracy |
| `cnn1d_accuracy_curve.png`, `cnn1d_loss_curve.png`, `cnn1d_confusion_matrix.png`, `cnn1d_roc_curve.png` | The four required figures |
| `cnn1d_probability_hist.png` | Extra figure: model confidence on the test set |
| `cnn1d_audit_length_outliers.png`, `cnn1d_eda_class_balance.png`, `cnn1d_eda_distinctive_words.png`, `cnn1d_eda_review_length.png` | Data audit and EDA figures |
| `cnn1d_seed_robustness.csv` | Run-to-run variation |
| `cnn1d_model.keras` | The trained model |
| `gru_short_model.keras`, `gru_short_extension_metrics.json` | Optional Step 13 extension model and its metrics (not part of the four-model comparison) |

**Comparison table row:** paste the row printed in Step 9. The overall best model can only be named after all four models have been evaluated under these same conditions.
**Training time** is only comparable between models if all four are timed on the same hardware.

## Try the model on your own reviews

Type or paste any hotel review and the trained models will say whether it looks **positive** or **negative**. The table shows the main GRU and the Step 13 extension side by side.
Your text goes through exactly the same steps as the training data:

1. **Cleaning**: `clean_text()` lowercases, removes HTML, URLs, digits and punctuation, and keeps negations such as "not".
2. **Tokenisation**: the cleaned text is split into words.
3. **Encoding**: each word becomes its id in the training vocabulary (words the model never saw become `<OOV>`).
4. **Padding**: the ids are padded or cut to `MAX_LEN` tokens, so every review has the same length.

Each model returns `P(positive)`, a probability between 0 and 1. A review is called *Positive* when it is at least the threshold (0.5).

Keep in mind that the models were trained on only 1,276 Chicago hotel reviews, each at least 25 words long (median about 130). Their probabilities are over-confident (see the error analysis), and mixed reviews, which praise one thing and criticise another, are the hardest cases.

**Very short texts.** The main GRU is unreliable for a review shorter than any training review: in our checks one-line reviews such as "Terrible hotel, dirty room and rude staff." were labelled *positive*. The function prints a note when this happens. The Step 13 extension was trained on short examples, so compare the two columns.

In [ ]:
from IPython.display import display


def predict_sentiment(model, texts, threshold=0.5):
    # `model` is one Keras model, or a dict {"name": model} to get one column per model side by side
    models = model if isinstance(model, dict) else {None: model}
    if isinstance(texts, str):
        texts = [texts]
    texts = list(texts)
    cleaned = [clean_text(t) for t in texts]
    # same steps as training: clean -> split into words -> encode with the training vocabulary -> pad
    X = pad([encode(c.split(), data.vocab) for c in cleaned])

    min_len = int(data.train_df["tokens"].map(len).min())      # the shortest original training review
    n_short = sum(len(c.split()) < min_len for c in cleaned)
    if n_short:
        print(f"Note: {n_short} review(s) have fewer than {min_len} words, shorter than any original training review. "
              "The main GRU is unreliable for very short text.")

    no_words = np.array([c == "" for c in cleaned])           # nothing left after cleaning (e.g. "???")
    probs = {name: m.predict(X, verbose=0).ravel() for name, m in models.items()}
    suffix = lambda name: "" if name is None else f" {name}"
    table = {"review": texts}
    for name, prob in probs.items():
        table["P(positive)" + suffix(name)] = np.where(no_words, np.nan, np.round(prob, 3))
    for name, prob in probs.items():
        label = np.where(prob >= threshold, "Positive", "Negative")
        table["prediction" + suffix(name)] = np.where(no_words, "No usable text", label)
    return pd.DataFrame(table)


models = {"1D CNN": model, "GRU + short": model_short}           # main model and the Step 13 extension

my_reviews = [
    # clearly positive
    "Absolutely wonderful stay! The staff were friendly and helpful, the room was spotless and very comfortable, and the view of the lake from our window was amazing. Breakfast was delicious and the location is perfect for exploring the city. I would happily come back and recommend it to friends.",
    # clearly negative
    "Terrible experience. The room was dirty, the carpet was stained and the bathroom smelled of mold. The staff were rude and did nothing when we complained, and the noise from the street kept us awake all night. It was the worst hotel I have ever stayed in and a waste of money.",
    # negation
    "Not good at all. I would not recommend this hotel to anyone. The room was not clean, the staff were not helpful and the breakfast was not worth the price. We did not get any apology when we complained, and we will not be coming back.",
    # mixed
    "Great location but the bed was uncomfortable and very noisy. We could walk to the shops and restaurants in a few minutes, which was perfect. Unfortunately the mattress was hard and the street noise made it hard to sleep, so we were tired the whole trip.",
    # mixed
    "The breakfast was excellent and the staff were lovely, however the room was tiny and smelled of smoke. The lobby and the restaurant looked great, but our room was old and the shower barely worked. Nice people, disappointing room.",
]

pd.set_option("display.max_colwidth", 90)
display(predict_sentiment(models, my_reviews))

# Very short reviews: the main GRU is unreliable here; the extension model was trained on short examples.
short_reviews = [
    "very bad place to visit never ever again visit to this place",
    "Not good at all.",
    "Worst hotel ever. Never again.",
    "Great hotel, lovely staff and a spotless room.",
]
display(predict_sentiment(models, short_reviews))

In [ ]:
# Interactive: type one review (leave empty to skip)
try:
    text = input("Enter a hotel review: ").strip()
except Exception:            # a non-interactive run (for example nbconvert) cannot ask for input, so skip
    text = ""
if text:
    display(predict_sentiment(models, [text]))